In [ ]:
# Configure SCBFM_ROOT_DIR and optionally SCBFM_FIGURE_DIR before launching Jupyter.
from pathlib import Path
import os
import sys

_candidates = [Path(os.environ['SCBFM_REPO_DIR'])] if os.environ.get('SCBFM_REPO_DIR') else []
_candidates += [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / 'src' / 'main.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the checkout or set SCBFM_REPO_DIR.')
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from notebook_setup import ROOT_DIR, OUTPUT_DIR, FIGURE_DIR


# Pretraining-corpus distributions

This local notebook renders the thesis figures and summary table from compact statistics generated on the cluster. It never loads the bulkRNA-seq or scRNA-seq expression matrices.

In [ ]:
from pathlib import Path
import json

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd


SCBFM_ROOT = REPO_ROOT
SUMMARY_ROOT = OUTPUT_DIR / 'distributions'
ARCHIVE_PATH = SUMMARY_ROOT / 'pretraining_distribution_histograms.npz'
STATISTICS_PATH = SUMMARY_ROOT / 'pretraining_distribution_statistics.csv'
METADATA_PATH = SUMMARY_ROOT / 'pretraining_distribution_metadata.json'
THESIS_FIGURES = FIGURE_DIR
THESIS_FIGURES.mkdir(parents=True, exist_ok=True)

MODALITIES = [('sc', 'scRNA-seq', '#356B9A'), ('bulk', 'bulkRNA-seq', '#C46A3A')]

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9.5,
    'xtick.labelsize': 8.5,
    'ytick.labelsize': 8.5,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})

print('Summary input:', SUMMARY_ROOT)
print('Figure output:', THESIS_FIGURES)

In [ ]:
required_paths = (ARCHIVE_PATH, STATISTICS_PATH, METADATA_PATH)
missing = [path for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Missing distribution summaries. Sync output/distributions from the '
        f'cluster first: {missing}'
    )

statistics = pd.read_csv(STATISTICS_PATH).set_index('modality')
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
with np.load(ARCHIVE_PATH, allow_pickle=False) as archive:
    histograms = {key: archive[key].copy() for key in archive.files}

expected_modalities = tuple(key for key, _, _ in MODALITIES)
observed_modalities = tuple(histograms['modality_keys'].astype(str))
if observed_modalities != expected_modalities:
    raise ValueError(
        f'Unexpected modality order {observed_modalities}; expected {expected_modalities}.'
    )
if int(histograms['format_version'][0]) != 1 or metadata.get('format_version') != 1:
    raise ValueError('Unsupported distribution-summary format version.')
for modality in expected_modalities:
    if histograms[f'log_count_histogram__{modality}'].sum() != statistics.loc[modality, 'nonzero_entries']:
        raise ValueError(f'Non-zero expression histogram total mismatch for {modality}.')
    if histograms[f'nonzero_gene_histogram__{modality}'].sum() != statistics.loc[modality, 'profiles']:
        raise ValueError(f'Non-zero gene histogram total mismatch for {modality}.')

print('Summary generated:', metadata['created_at_utc'])
display(statistics)

## Summary statistics

In [ ]:
statistic_rows = [
    ('Profiles', 'profiles', lambda value: f'{int(value):,}'),
    ('Genes per profile', 'genes', lambda value: f'{int(value):,}'),
    ('Zero entries', 'zero_fraction', lambda value: f'{value:.1%}'),
    ('Median non-zero expression value', 'median_nonzero_count', lambda value: f'{value:,.1f}'),
    ('Non-zero values equal to 1', 'fraction_nonzero_equal_one', lambda value: f'{value:.1%}'),
    ('Median total count per profile', 'median_total_count_per_profile', lambda value: f'{value:,.1f}'),
    ('Mean non-zero genes per profile', 'mean_nonzero_genes_per_profile', lambda value: f'{value:,.1f}'),
    ('Median non-zero genes per profile', 'median_nonzero_genes_per_profile', lambda value: f'{value:,.1f}'),
]
summary_table = pd.DataFrame(
    {
        label: [formatter(statistics.loc[key, field]) for key, _, _ in MODALITIES]
        for label, field, formatter in statistic_rows
    },
    index=[label for _, label, _ in MODALITIES],
).T
summary_table.index.name = 'Statistic'
display(summary_table)

for key, label, _ in MODALITIES:
    value = statistics.loc[key, 'median_total_count_per_profile']
    print(f'Median total count per profile ({label}): {value:,.1f}')

latex_summary_table = summary_table.replace({'%': r'\%'}, regex=True)
print('\nLaTeX tabular output:\n')
print(latex_summary_table.to_latex(escape=False, column_format='lcc'))

## Thesis figures

In [ ]:
def style_distribution_axes(axes):
    for ax in axes:
        ax.grid(axis='y', color='#E5E5E5', linewidth=0.65)
        ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        for side in ('left', 'bottom'):
            ax.spines[side].set_color('#555555')
            ax.spines[side].set_linewidth(0.8)
        ax.tick_params(axis='both', length=3, width=0.8, color='#555555')


def save_figure(fig, stem):
    pdf_path = THESIS_FIGURES / f'{stem}.pdf'
    png_path = THESIS_FIGURES / f'{stem}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
    print('Saved:', pdf_path)
    print('Saved:', png_path)


def plot_broken_histogram_pair(
    *, edges_key, histogram_prefix, xlabel, ylabel, stem,
    lower_ylim=(0, 20), upper_ylim=(50, 60),
):
    edges = histograms[edges_key]
    fig, axes = plt.subplots(
        2, 2, figsize=(7.2, 3.55), sharex='col', sharey='row',
        gridspec_kw={'height_ratios': (1, 2.4), 'hspace': 0.05},
    )
    top_axes, bottom_axes = axes
    for column, (key, label, color) in enumerate(MODALITIES):
        counts = histograms[f'{histogram_prefix}__{key}']
        percentages = 100 * counts / counts.sum()
        for ax in (top_axes[column], bottom_axes[column]):
            ax.stairs(
                percentages, edges, fill=True, color=color,
                alpha=0.85, linewidth=0.7,
            )
            ax.set_xlim(edges[0], edges[-1])
        top_axes[column].set_title(label, color=color, pad=7)
        bottom_axes[column].set_xlabel(xlabel, labelpad=6)

    for ax in top_axes:
        ax.set_ylim(*upper_ylim)
        ax.spines['bottom'].set_visible(False)
        ax.tick_params(axis='x', bottom=False, labelbottom=False)
    for ax in bottom_axes:
        ax.set_ylim(*lower_ylim)
        ax.spines['top'].set_visible(False)

    style_distribution_axes(axes.ravel())
    for ax in top_axes:
        ax.spines['bottom'].set_visible(False)
    for ax in bottom_axes:
        ax.spines['top'].set_visible(False)
    top_axes[0].set_yticks([55, 60])
    bottom_axes[0].set_yticks([0, 5, 10, 15, 20])
    top_axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    bottom_axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))

    break_size = 0.012
    break_style = {'color': '#555555', 'clip_on': False, 'linewidth': 0.8}
    for top_ax, bottom_ax in zip(top_axes, bottom_axes):
        top_ax.plot(
            (-break_size, break_size), (-break_size, break_size),
            transform=top_ax.transAxes, **break_style,
        )
        bottom_ax.plot(
            (-break_size, break_size), (1 - break_size, 1 + break_size),
            transform=bottom_ax.transAxes, **break_style,
        )

    fig.supylabel(ylabel, x=0.015, fontsize=9.5)
    fig.subplots_adjust(left=0.105, right=0.985, bottom=0.16, top=0.90, wspace=0.10)
    save_figure(fig, stem)
    return fig


def plot_histogram_pair(*, edges_key, histogram_prefix, xlabel, ylabel, stem, xticks=None):
    edges = histograms[edges_key]
    fig, axes = plt.subplots(
        1, 2, figsize=(7.2, 3.15), sharex=True, sharey=True, layout='constrained'
    )
    for ax, (key, label, color) in zip(axes, MODALITIES):
        counts = histograms[f'{histogram_prefix}__{key}']
        percentages = 100 * counts / counts.sum()
        ax.stairs(
            percentages, edges, fill=True, color=color, alpha=0.85, linewidth=0.7
        )
        ax.set_title(label, color=color, pad=7)
        ax.set_xlabel(xlabel, labelpad=6)
        ax.set_xlim(edges[0], edges[-1])
        if xticks is not None:
            ax.set_xticks(xticks)
    axes[0].set_ylabel(ylabel, labelpad=7)
    axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    style_distribution_axes(axes)
    save_figure(fig, stem)
    return fig

In [ ]:
expression_figure = plot_broken_histogram_pair(
    edges_key='log_count_edges',
    histogram_prefix='log_count_histogram',
    xlabel='log1p(expression value)',
    ylabel='Non-zero entries (%)',
    stem='pretraining_nonzero_expression_distribution',
)
plt.show()

In [ ]:
gene_figure = plot_histogram_pair(
    edges_key='nonzero_gene_edges',
    histogram_prefix='nonzero_gene_histogram',
    xlabel='Non-zero genes per profile',
    ylabel='Profiles (%)',
    stem='pretraining_nonzero_genes_distribution',
    xticks=[0, 4_000, 8_000, 12_000],
)
plt.show()

The two panels within each figure use identical histogram bins and shared axes. Percentages rather than absolute frequencies are shown because the two pretraining corpora contain different numbers of profiles and non-zero entries.